In [68]:
!pip install --quiet --upgrade openai duckdb pandas matplotlib

In [2]:
from openai import OpenAI
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import getpass
import re

In [3]:
client = OpenAI(api_key=getpass.getpass("Enter your OpenAI API key: "))

Enter your OpenAI API key: ··········


In [32]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("umuttuygurr/e-commerce-customer-behavior-and-sales-analysis-tr")
print("Path to dataset files:", path)

Path to dataset files: /root/.cache/kagglehub/datasets/umuttuygurr/e-commerce-customer-behavior-and-sales-analysis-tr/versions/1


In [36]:
df = pd.read_csv(f"{path}/ecommerce_customer_behavior_dataset.csv")
#/root/.cache/kagglehub/datasets/umuttuygurr/e-commerce-customer-behavior-and-sales-analysis-tr/versions/1/ecommerce_customer_behavior_dataset.csv
print("Rows:", len(df))
df.head(3)

Rows: 5000


,Order_ID,Customer_ID,Date,Age,Gender,City,Product_Category,Unit_Price,Quantity,Discount_Amount,Total_Amount,Payment_Method,Device_Type,Session_Duration_Minutes,Pages_Viewed,Is_Returning_Customer,Delivery_Time_Days,Customer_Rating
0,ORD_001337,CUST_01337,2023-01-01,27,Female,Bursa,Toys,54.28,1,0.0,54.28,Debit Card,Mobile,4,14,True,8,5
1,ORD_004885,CUST_04885,2023-01-01,42,Male,Konya,Toys,244.90,1,0.0,244.90,Credit Card,Mobile,11,3,True,3,3
2,ORD_004507,CUST_04507,2023-01-01,43,Female,Ankara,Food,48.15,5,0.0,240.75,Credit Card,Mobile,7,8,True,5,2


In [37]:
#Load table to DuckDB
con = duckdb.connect(database=':memory:')
con.execute("CREATE TABLE orders AS SELECT * FROM df")
con.sql("SELECT * from orders limit 5").df()

,Order_ID,Customer_ID,Date,Age,Gender,City,Product_Category,Unit_Price,Quantity,Discount_Amount,Total_Amount,Payment_Method,Device_Type,Session_Duration_Minutes,Pages_Viewed,Is_Returning_Customer,Delivery_Time_Days,Customer_Rating
0,ORD_001337,CUST_01337,2023-01-01,27,Female,Bursa,Toys,54.28,1,0.00,54.28,Debit Card,Mobile,4,14,True,8,5
1,ORD_004885,CUST_04885,2023-01-01,42,Male,Konya,Toys,244.90,1,0.00,244.90,Credit Card,Mobile,11,3,True,3,3
2,ORD_004507,CUST_04507,2023-01-01,43,Female,Ankara,Food,48.15,5,0.00,240.75,Credit Card,Mobile,7,8,True,5,2
3,ORD_000645,CUST_00645,2023-01-01,32,Male,Istanbul,Electronics,804.06,1,229.28,574.78,Credit Card,Mobile,8,10,False,1,4
4,ORD_000690,CUST_00690,2023-01-01,40,Female,Istanbul,Sports,755.61,5,0.00,3778.05,Cash on Delivery,Desktop,21,10,True,7,4


In [63]:
##LLM Section
# Format the returned query
def fix_duckdb_sql(sql: str) -> str:
    """
    Cleans and corrects common LLM SQL formatting issues for DuckDB.
    Handles:
      - UNION / UNION ALL wrapping
      - misplaced 'ALL' tokens
      - trailing semicolons
      - stray English text or comments
    """
    # Remove English explanations accidentally included by model
    sql = re.sub(r'(?i)(^to\s|get\s|return\s).*', '', sql)

    # Fix "UNION (ALL SELECT" -> "UNION ALL (SELECT"
    sql = re.sub(r'UNION\s*\(\s*ALL', 'UNION ALL (', sql, flags=re.IGNORECASE)

    # Fix cases where ALL is misplaced
    sql = sql.replace("UNION ( ALL", "UNION ALL (")
    sql = sql.replace("UNION( ALL", "UNION ALL (")

    # Ensure subqueries around UNIONs are parenthesized
    if "UNION" in sql.upper():
        parts = re.split(r'\bUNION(?: ALL)?\b', sql, flags=re.IGNORECASE)
        union_tokens = re.findall(r'\bUNION(?: ALL)?\b', sql, flags=re.IGNORECASE)
        wrapped_parts = []
        for p in parts:
            p = p.strip().rstrip(";")
            if not (p.startswith("(") and p.endswith(")")):
                p = f"(\n{p}\n)"
            wrapped_parts.append(p)
        # Reassemble correctly
        sql = " ".join(
            f"{wrapped_parts[i]} {union_tokens[i]}" if i < len(union_tokens) else wrapped_parts[i]
            for i in range(len(wrapped_parts))
        )

    # Remove any stray trailing semicolon
    sql = sql.strip().rstrip(";")

    return sql.strip()

In [291]:
# Text → SQL translator
def text_to_sql(user_query, table_schema):
    prompt = f"""
You are an expert SQL analyst. Use the conversation history and the user's question
to generate a valid DuckDB SQL query on the 'orders' table.

Rules:
- Return ONLY SQL (no markdown, no explanation)
- Always cast date/time strings before using date functions, e.g., CAST(Date AS TIMESTAMP)
- When displaying currency fields (like Total_Amount, Sales, Revenue, or similar),
  limit it to 2 decimals
- Do not use CTE For a simple query lets say finding the total count & sum
- Use CTEs (WITH ... AS) ONLY when:
  - A direct query is not possible, or
  - You need to reference the same data multiple times in the query, or
  - You need multiple layers of aggregation (aggregation over aggregation), or
  - When asked to find the "highest" or "maximum" per group (e.g., best-selling day for each product),
    generate a query that returns only that row per group, using WHERE ... IN (SELECT ... MAX(...) GROUP BY ...),
    not all intermediate results.
- Always generate DuckDB-compliant syntax.

- Use columns:
{schema}
User question: "{user_query}"

SQL:
"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    # Clean output — remove markdown if any
    sql_query = response.choices[0].message.content.strip()
    sql_query_refined = fix_duckdb_sql(sql_query)

    return sql_query_refined

In [292]:
schema = con.sql("DESCRIBE orders").df().to_string(index=False)
print(schema)

             column_name column_type null  key default extra
                Order_ID     VARCHAR  YES None    None  None
             Customer_ID     VARCHAR  YES None    None  None
                    Date     VARCHAR  YES None    None  None
                     Age      BIGINT  YES None    None  None
                  Gender     VARCHAR  YES None    None  None
                    City     VARCHAR  YES None    None  None
        Product_Category     VARCHAR  YES None    None  None
              Unit_Price      DOUBLE  YES None    None  None
                Quantity      BIGINT  YES None    None  None
         Discount_Amount      DOUBLE  YES None    None  None
            Total_Amount      DOUBLE  YES None    None  None
          Payment_Method     VARCHAR  YES None    None  None
             Device_Type     VARCHAR  YES None    None  None
Session_Duration_Minutes      BIGINT  YES None    None  None
            Pages_Viewed      BIGINT  YES None    None  None
   Is_Returning_Customer

In [293]:
#Chatbot logic
def chatbot(query):
    print(f"🔹 User: {query}")
    sql_query = text_to_sql(query, schema)
    print(f"\n🧠 Generated SQL:\n{sql_query}")
    try:
        result = con.sql(sql_query).df()
        display(result)
    except Exception as e:
        print("⚠️ Error executing SQL:", e)

In [295]:
chatbot("what is the total count of orders & sales amount for each category. provide only the top 5 sale amount")

🔹 User: what is the total count of orders & sales amount for each category. provide only the top 5 sale amount

🧠 Generated SQL:
SELECT Product_Category, 
       COUNT(Order_ID) AS Total_Orders, 
       ROUND(SUM(Total_Amount), 2) AS Total_Sales 
FROM orders 
GROUP BY Product_Category 
ORDER BY Total_Sales DESC 
LIMIT 5


,Product_Category,Total_Orders,Total_Sales
0,Electronics,624,2328806.81
1,Home & Garden,621,908348.86
2,Sports,667,754563.56
3,Fashion,622,375214.93
4,Toys,610,223142.48


In [296]:
chatbot("What are the top 5 regions based on sales amount")

🔹 User: What are the top 5 regions based on sales amount

🧠 Generated SQL:
SELECT City, ROUND(SUM(Total_Amount), 2) AS Total_Sales
FROM orders
GROUP BY City
ORDER BY Total_Sales DESC
LIMIT 5


,City,Total_Sales
0,Istanbul,1334122.56
1,Ankara,657535.82
2,Izmir,567534.67
3,Bursa,459076.31
4,Adana,427059.63


In [297]:
chatbot("what payment mode has the largest preference?")

🔹 User: what payment mode has the largest preference?

🧠 Generated SQL:
SELECT Payment_Method, COUNT(*) AS Preference_Count
FROM orders
GROUP BY Payment_Method
ORDER BY Preference_Count DESC
LIMIT 1


,Payment_Method,Preference_Count
0,Credit Card,2012


In [ ]:
#Lets validate the results by running the query manually

In [306]:
chatbot("Which day of the week has the highest sales for each product")

🔹 User: Which day of the week has the highest sales for each product

🧠 Generated SQL:
WITH Sales_Per_Day AS (
    SELECT 
        Product_Category,
        CAST(Date AS TIMESTAMP) AS Sale_Date,
        SUM(Total_Amount) AS Total_Sales
    FROM 
        orders
    GROUP BY 
        Product_Category, CAST(Date AS TIMESTAMP)
),
Max_Sales AS (
    SELECT 
        Product_Category,
        MAX(Total_Sales) AS Max_Sales
    FROM 
        Sales_Per_Day
    GROUP BY 
        Product_Category
)
SELECT 
    spd.Product_Category,
    spd.Sale_Date,
    ROUND(spd.Total_Sales, 2) AS Total_Sales
FROM 
    Sales_Per_Day spd
JOIN 
    Max_Sales ms ON spd.Product_Category = ms.Product_Category AND spd.Total_Sales = ms.Max_Sales


,Product_Category,Sale_Date,Total_Sales
0,Fashion,2023-08-29,5441.31
1,Books,2023-09-01,1363.81
2,Electronics,2023-12-04,42338.06
3,Sports,2023-12-11,12250.72
4,Beauty,2024-01-21,2294.31
5,Toys,2024-03-09,4031.92
6,Food,2023-10-09,2297.15
7,Home & Garden,2024-01-29,16426.64


In [246]:
chatbot("What is the return rate % per category? I want the results in desc order")

🔹 User: What is the return rate % per category? I want the results in desc order

🧠 Generated SQL:
SELECT 
    Product_Category, 
    ROUND(SUM(CASE WHEN Is_Returning_Customer THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Return_Rate_Percentage
FROM 
    orders
GROUP BY 
    Product_Category
ORDER BY 
    Return_Rate_Percentage DESC


,Product_Category,Return_Rate_Percentage
0,Electronics,61.86
1,Food,61.39
2,Home & Garden,59.58
3,Toys,59.51
4,Fashion,59.49
5,Books,59.25
6,Sports,59.22
7,Beauty,58.13


In [238]:
chatbot("Provide the top 5 products along with its order details with poor ratings (<3) ?")

🔹 User: Provide the top 5 products along with its order details with poor ratings (<3) ?

🧠 Generated SQL:
SELECT Product_Category, Order_ID, Customer_ID, CAST(Date AS TIMESTAMP) AS Order_Date, 
       Age, Gender, City, Unit_Price, Quantity, 
       Discount_Amount, ROUND(Total_Amount, 2) AS Total_Amount, 
       Payment_Method, Device_Type, Session_Duration_Minutes, 
       Pages_Viewed, Is_Returning_Customer, Delivery_Time_Days, 
       Customer_Rating
FROM orders
WHERE Customer_Rating < 3
ORDER BY Total_Amount DESC
LIMIT 5


,Product_Category,Order_ID,Customer_ID,Order_Date,Age,Gender,City,Unit_Price,Quantity,Discount_Amount,Total_Amount,Payment_Method,Device_Type,Session_Duration_Minutes,Pages_Viewed,Is_Returning_Customer,Delivery_Time_Days,Customer_Rating
0,Electronics,ORD_004019,CUST_04019,2023-05-30,28,Male,Ankara,3227.59,5,0.00,16137.95,Credit Card,Mobile,13,15,False,10,2
1,Electronics,ORD_003467,CUST_03467,2023-12-11,30,Male,Kayseri,3145.83,5,0.00,15729.15,Digital Wallet,Desktop,23,3,False,4,2
2,Electronics,ORD_000316,CUST_00316,2023-12-04,35,Female,Bursa,4047.49,4,735.59,15454.37,Credit Card,Tablet,20,7,True,11,2
3,Electronics,ORD_002068,CUST_02068,2023-01-29,33,Female,Izmir,5164.64,3,1394.26,14099.66,Debit Card,Mobile,11,8,True,5,1
4,Electronics,ORD_004758,CUST_04758,2024-02-04,50,Male,Ankara,3440.00,4,0.00,13760.00,Digital Wallet,Mobile,13,7,False,7,1


In [239]:
chatbot(" Which device is most preffered by customers for placing orders?")

🔹 User:  Which device is most preffered by customers for placing orders?

🧠 Generated SQL:
SELECT Device_Type, COUNT(*) AS Order_Count
FROM orders
GROUP BY Device_Type
ORDER BY Order_Count DESC
LIMIT 1


,Device_Type,Order_Count
0,Mobile,2795


In [ ]:
#Validation
# Try running the query in the below format for cross verify the results

In [120]:
con.sql("""
 SELECT
        Payment_Method,
        COUNT(*) AS Payment_Count
    FROM
        orders
    GROUP BY
        Payment_Method
        """)

┌──────────────────┬───────────────┐
│  Payment_Method  │ Payment_Count │
│     varchar      │     int64     │
├──────────────────┼───────────────┤
│ Debit Card       │          1265 │
│ Digital Wallet   │           965 │
│ Bank Transfer    │           510 │
│ Credit Card      │          2012 │
│ Cash on Delivery │           248 │
└──────────────────┴───────────────┘